# BanglaWriting: Bangla Handwritten Word Recognition (CRNN + CTC)

**Dataset:** BanglaWriting — A multi-purpose offline Bangla handwriting dataset  
**DOI:** 10.17632/r43wkvdk4w.1  
**Task:** Handwritten Bangla word recognition using CRNN (CNN + BiLSTM + CTC Loss)

### Dataset Stats
- 260 full-page handwriting images from different individuals
- 21,234 word-level bounding boxes with Unicode labels
- 124 unique characters (Bangla script + digits + punctuation)
- 5,470 unique words

### Architecture
- **CNN Backbone:** Feature extraction from word images
- **BiLSTM:** Sequence modeling over feature columns
- **CTC Loss:** Connectionist Temporal Classification for variable-length output

### Kaggle Setup
Upload `converted.zip` as a Kaggle dataset, then set the input path below.

In [ ]:
# ============================================================
# 1. Import Required Libraries
# ============================================================
import os
import json
import glob
import gc
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms

# Mixed precision aliases (works with both old and new PyTorch API)
autocast = lambda **kwargs: torch.amp.autocast('cuda', **kwargs)
GradScaler = lambda **kwargs: torch.amp.GradScaler('cuda', **kwargs)

# --- Install Bengali font for Matplotlib ---
import urllib.request
FONT_URL = "https://github.com/google/fonts/raw/main/ofl/notosansbengali/NotoSansBengali%5Bwdth%2Cwght%5D.ttf"
FONT_PATH = "/tmp/NotoSansBengali.ttf"
if not os.path.exists(FONT_PATH):
    urllib.request.urlretrieve(FONT_URL, FONT_PATH)
    print(f"Downloaded Bengali font to {FONT_PATH}")
fm.fontManager.addfont(FONT_PATH)
BANGLA_FONT = fm.FontProperties(fname=FONT_PATH)
plt.rcParams['font.family'] = BANGLA_FONT.get_name()
print(f"Bengali font set: {BANGLA_FONT.get_name()}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("All libraries imported successfully!")

## 2. Check GPU Availability and Configuration

In [ ]:
# ============================================================
# 2. GPU Check and Configuration
# ============================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1024**3:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"cuDNN Version: {torch.backends.cudnn.version()}")
    torch.cuda.empty_cache()
else:
    print("WARNING: No GPU detected! Training will be very slow on CPU.")
    print("Please enable GPU: Settings > Accelerator > GPU (P100/T4)")

## 3. Configuration & Paths

Set the dataset path. On Kaggle, upload `converted.zip` as a dataset and use the `/kaggle/input/` path.

In [ ]:
# ============================================================
# 3. Configuration
# ============================================================

# --- PATH CONFIGURATION ---
# On Kaggle: Upload converted.zip as dataset named "banglawriting"
OUTPUT_DIR = "/kaggle/working"

# Debug: show what's available under /kaggle/input/
if os.path.exists("/kaggle/input"):
    print("Kaggle input contents:")
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.replace("/kaggle/input", "").count(os.sep)
        if depth < 3:  # Only show 3 levels deep
            indent = "  " * depth
            print(f"{indent}{os.path.basename(root)}/  ({len(files)} files)")

# Auto-detect data directory across environments
DATA_DIR = None
candidates = [
    "/kaggle/input/banglawriting/converted/converted",
    "/kaggle/input/banglawriting/converted",
    "/kaggle/input/banglawriting",
    "converted/converted",
    "../converted/converted",
    "/home/rohan/Softograph/bangla_dataset_1_download/converted/converted",
]

# Also scan all subdirs under /kaggle/input/ dynamically
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if any(f.endswith('.json') for f in files):
            candidates.insert(0, root)  # Prioritize found paths

for candidate in candidates:
    if os.path.exists(candidate) and any(f.endswith('.json') for f in os.listdir(candidate)):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find BanglaWriting dataset!\n"
        "On Kaggle: upload converted.zip as a dataset, then check the path above."
    )

if not os.path.exists(OUTPUT_DIR):
    OUTPUT_DIR = "."

print(f"\nData directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Files found: {len(os.listdir(DATA_DIR))}")

# --- HYPERPARAMETERS ---
IMG_HEIGHT = 64          # Fixed height for word images
IMG_WIDTH = 256          # Max width (padded)
BATCH_SIZE = 64          # Fits well on Kaggle P100/T4 (16GB)
NUM_EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2          # Kaggle has 2 CPUs
VAL_SPLIT = 0.15         # 15% validation

print(f"\nHyperparameters:")
print(f"  Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")

## 4. Load and Explore the Dataset

Parse all JSON annotations to extract word-level crops (bounding boxes + Unicode labels) from the full-page handwriting images.

In [ ]:
# ============================================================
# 4. Load Dataset - Parse annotations and build samples
# ============================================================

def find_image_path(data_dir, image_name):
    """Try multiple strategies to locate the image file."""
    basename = os.path.basename(image_name)
    name_no_ext = os.path.splitext(basename)[0]
    
    # Build list of directories to search: data_dir, parent, siblings
    search_dirs = [data_dir]
    parent = os.path.dirname(data_dir)
    if parent:
        search_dirs.append(parent)
        # Also check sibling directories
        if os.path.isdir(parent):
            for d in os.listdir(parent):
                full = os.path.join(parent, d)
                if os.path.isdir(full) and full != data_dir:
                    search_dirs.append(full)
    
    for d in search_dirs:
        for ext in ['', '.jpg', '.JPG', '.jpeg', '.png']:
            if ext == '':
                path = os.path.join(d, basename)
            else:
                path = os.path.join(d, name_no_ext + ext)
            if os.path.exists(path):
                return path
    return None

def load_banglawriting_dataset(data_dir):
    samples = []
    skipped = 0
    json_files = sorted(glob.glob(os.path.join(data_dir, "*.json")))
    print(f"Found {len(json_files)} annotation files")
    
    # Check what image files actually exist in data_dir and parent
    img_files = [f for f in os.listdir(data_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f"Image files in data_dir: {len(img_files)}")
    
    # Also check parent dir
    parent = os.path.dirname(data_dir)
    if parent and os.path.isdir(parent):
        parent_imgs = [f for f in os.listdir(parent) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"Image files in parent dir ({parent}): {len(parent_imgs)}")
        # Check sibling dirs too
        for d in os.listdir(parent):
            full = os.path.join(parent, d)
            if os.path.isdir(full) and full != data_dir:
                sib_imgs = [f for f in os.listdir(full) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
                if sib_imgs:
                    print(f"Image files in sibling dir ({full}): {len(sib_imgs)}")
    
    for jf in json_files:
        with open(jf, 'r', encoding='utf-8') as fp:
            data = json.load(fp)
        
        img_path = find_image_path(data_dir, data['imagePath'])
        if img_path is None:
            skipped += 1
            if skipped <= 3:
                print(f"  WARNING: Image not found for '{data['imagePath']}' (JSON: {os.path.basename(jf)})")
            continue
        
        img_h = data['imageHeight']
        img_w = data['imageWidth']
        
        basename = os.path.splitext(os.path.basename(data['imagePath']))[0]
        parts = basename.split('_')
        writer_id = int(parts[0]) if len(parts) > 0 and parts[0].isdigit() else hash(basename) % 100000
        age = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
        gender = int(parts[2]) if len(parts) > 2 and parts[2].isdigit() else 0
        
        for shape in data['shapes']:
            label = shape['label'].strip()
            if not label:
                continue
            
            pts = shape['points']
            x1, y1 = int(pts[0][0]), int(pts[0][1])
            x2, y2 = int(pts[1][0]), int(pts[1][1])
            x1, x2 = min(x1, x2), max(x1, x2)
            y1, y2 = min(y1, y2), max(y1, y2)
            x1 = max(0, x1)
            y1 = max(0, y1)
            x2 = min(img_w, x2)
            y2 = min(img_h, y2)
            
            if x2 - x1 < 5 or y2 - y1 < 5:
                continue
            
            samples.append({
                'img_path': img_path,
                'bbox': [x1, y1, x2, y2],
                'label': label,
                'writer_id': writer_id,
                'age': age,
                'gender': gender,
            })
    
    if skipped > 0:
        print(f"\nWARNING: Skipped {skipped}/{len(json_files)} JSON files (images not found)")
    
    # Verify stored paths
    unique_paths = set(s['img_path'] for s in samples)
    missing = [p for p in unique_paths if not os.path.exists(p)]
    print(f"\nVerification: {len(unique_paths)} unique images, {len(missing)} missing")
    
    return samples

all_samples = load_banglawriting_dataset(DATA_DIR)
print(f"\nTotal word samples: {len(all_samples)}")

if len(all_samples) == 0:
    raise RuntimeError(
        "No samples loaded! The dataset images (.jpg) may not be uploaded.\n"
        "Make sure converted.zip includes BOTH .json AND .jpg files."
    )

df = pd.DataFrame(all_samples)
print(f"\nDataset shape: {df.shape}")
df['label_len'] = df['label'].str.len()
print(f"\nLabel length stats:")
print(df['label_len'].describe())
print(f"\nUnique writers: {df['writer_id'].nunique()}")
print(f"Unique words: {df['label'].nunique()}")
print(f"Gender: Male={len(df[df['gender']==0])}, Female={len(df[df['gender']==1])}")

## 5. Build Character Vocabulary

In [ ]:
# ============================================================
# 5. Build Character Vocabulary
# ============================================================

def build_vocab(samples):
    char_counter = Counter()
    for s in samples:
        for ch in s['label']:
            char_counter[ch] += 1
    
    chars = sorted(char_counter.keys())
    
    # Index 0 is reserved for CTC blank token
    char_to_idx = {ch: idx + 1 for idx, ch in enumerate(chars)}
    idx_to_char = {idx + 1: ch for idx, ch in enumerate(chars)}
    idx_to_char[0] = '<blank>'
    
    num_classes = len(chars) + 1  # +1 for CTC blank
    
    print(f"Vocabulary size: {len(chars)} characters + 1 blank = {num_classes} classes")
    print(f"\nCharacter frequency (top 20):")
    for ch, cnt in char_counter.most_common(20):
        print(f"  '{ch}' (U+{ord(ch):04X}): {cnt}")
    
    return char_to_idx, idx_to_char, num_classes

char_to_idx, idx_to_char, NUM_CLASSES = build_vocab(all_samples)
print(f"\nTotal classes (including CTC blank): {NUM_CLASSES}")

In [ ]:
# ============================================================
# 6. Visualize sample word crops
# ============================================================

def crop_word_image(img_path, bbox):
    if not os.path.exists(img_path):
        return None
    img = Image.open(img_path).convert('RGB')
    x1, y1, x2, y2 = bbox
    return img.crop((x1, y1, x2, y2))

fig, axes = plt.subplots(4, 4, figsize=(16, 8))
sample_indices = random.sample(range(len(all_samples)), min(64, len(all_samples)))

plot_idx = 0
for idx in sample_indices:
    if plot_idx >= 16:
        break
    s = all_samples[idx]
    crop = crop_word_image(s['img_path'], s['bbox'])
    if crop is None:
        continue
    ax = axes.flatten()[plot_idx]
    ax.imshow(crop)
    ax.set_title(s['label'], fontsize=10)
    ax.axis('off')
    plot_idx += 1

# Hide unused axes
for i in range(plot_idx, 16):
    axes.flatten()[i].axis('off')

plt.suptitle("Sample Word Crops from BanglaWriting Dataset", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_crops.png"), dpi=100, bbox_inches='tight')
plt.show()
print(f"Plotted {plot_idx} samples")

## 7. Data Preprocessing & Train/Val Split

Split by **writer** (not by sample) to avoid data leakage — validation writers are unseen during training.

In [ ]:
# ============================================================
# 7. Split data by writer to avoid leakage
# ============================================================

writer_ids = list(set(s['writer_id'] for s in all_samples))
print(f"Total unique writers: {len(writer_ids)}")

train_writers, val_writers = train_test_split(
    writer_ids, test_size=VAL_SPLIT, random_state=SEED
)

print(f"Train writers: {len(train_writers)}")
print(f"Val writers: {len(val_writers)}")

train_samples = [s for s in all_samples if s['writer_id'] in set(train_writers)]
val_samples = [s for s in all_samples if s['writer_id'] in set(val_writers)]

print(f"\nTrain samples: {len(train_samples)}")
print(f"Val samples: {len(val_samples)}")
print(f"Val ratio: {len(val_samples) / len(all_samples):.2%}")

## 8. Create PyTorch Dataset & DataLoaders with GPU Optimization

In [ ]:
# ============================================================
# 8. PyTorch Dataset for BanglaWriting word recognition
# ============================================================

class BanglaWordDataset(Dataset):
    def __init__(self, samples, char_to_idx, img_height=64, img_width=256, augment=False):
        self.samples = samples
        self.char_to_idx = char_to_idx
        self.img_height = img_height
        self.img_width = img_width
        self.augment = augment
        self._img_cache = {}
        
        if augment:
            self.transform = transforms.Compose([
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.RandomAffine(degrees=2, translate=(0.02, 0.02), 
                                        scale=(0.95, 1.05)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5], std=[0.5]),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.5], std=[0.5]),
            ])
    
    def __len__(self):
        return len(self.samples)
    
    def _get_full_image(self, img_path):
        if img_path not in self._img_cache:
            self._img_cache[img_path] = Image.open(img_path).convert('L')
        return self._img_cache[img_path]
    
    def _resize_and_pad(self, img):
        w, h = img.size
        new_h = self.img_height
        new_w = max(1, int(w * new_h / h))
        if new_w > self.img_width:
            new_w = self.img_width
        img = img.resize((new_w, new_h), Image.BILINEAR)
        padded = Image.new('L', (self.img_width, self.img_height), 255)
        padded.paste(img, (0, 0))
        return padded, new_w
    
    def _encode_label(self, label):
        return [self.char_to_idx[ch] for ch in label if ch in self.char_to_idx]
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        full_img = self._get_full_image(sample['img_path'])
        x1, y1, x2, y2 = sample['bbox']
        word_img = full_img.crop((x1, y1, x2, y2))
        word_img, actual_width = self._resize_and_pad(word_img)
        word_img = self.transform(word_img)
        label = self._encode_label(sample['label'])
        label_tensor = torch.tensor(label, dtype=torch.long)
        return word_img, label_tensor, len(label), actual_width


def collate_fn(batch):
    images, labels, label_lengths, widths = zip(*batch)
    images = torch.stack(images, 0)
    max_label_len = max(label_lengths)
    padded_labels = torch.zeros(len(labels), max_label_len, dtype=torch.long)
    for i, (label, length) in enumerate(zip(labels, label_lengths)):
        padded_labels[i, :length] = label
    label_lengths = torch.tensor(label_lengths, dtype=torch.long)
    return images, padded_labels, label_lengths

print("Dataset class defined!")

In [ ]:
# ============================================================
# 9. Create DataLoaders
# ============================================================

train_dataset = BanglaWordDataset(
    train_samples, char_to_idx, 
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH, augment=True
)
val_dataset = BanglaWordDataset(
    val_samples, char_to_idx,
    img_height=IMG_HEIGHT, img_width=IMG_WIDTH, augment=False
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
    drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn,
    drop_last=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Sanity check
batch = next(iter(train_loader))
imgs, labels, label_lens = batch
print(f"\nBatch shapes:")
print(f"  Images: {imgs.shape}")
print(f"  Labels: {labels.shape}")
print(f"  Label lengths: {label_lens}")

## 9. Model Architecture: CRNN (CNN + BiLSTM + CTC)

The model has three stages:
1. **CNN Feature Extractor**: Extracts visual features from word images
2. **Bidirectional LSTM**: Models sequential dependencies across feature columns
3. **Linear Projection**: Maps to character classes for CTC decoding

In [ ]:
# ============================================================
# 10. CRNN Model Definition
# ============================================================

class BidirectionalLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)
    
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out)


class CRNN(nn.Module):
    def __init__(self, num_classes, img_height=64, hidden_size=256):
        super().__init__()
        
        self.cnn = nn.Sequential(
            # Block 1: 1 -> 64
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            # Block 2: 64 -> 128
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            # Block 3: 128 -> 256
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            # Block 4: 256 -> 256
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            # Block 5: 256 -> 512
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            # Block 6: 512 -> 512
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            # Block 7: 512 -> 512
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
        )
        
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))
        
        self.rnn = nn.Sequential(
            BidirectionalLSTM(512, hidden_size, hidden_size),
            nn.Dropout(0.3),
            BidirectionalLSTM(hidden_size, hidden_size, num_classes),
        )
    
    def forward(self, x):
        conv = self.cnn(x)
        conv = self.adaptive_pool(conv)
        conv = conv.squeeze(2)
        conv = conv.permute(0, 2, 1)
        output = self.rnn(conv)
        output = output.permute(1, 0, 2)
        output = F.log_softmax(output, dim=2)
        return output


model = CRNN(num_classes=NUM_CLASSES, img_height=IMG_HEIGHT, hidden_size=256)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: CRNN")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB (float32)")

with torch.no_grad():
    dummy = torch.randn(2, 1, IMG_HEIGHT, IMG_WIDTH).to(device)
    out = model(dummy)
    print(f"\nForward pass test:")
    print(f"  Input:  {dummy.shape}")
    print(f"  Output: {out.shape} (T={out.shape[0]}, B={out.shape[1]}, C={out.shape[2]})")

## 10. Configure Training: Optimizer, Scheduler, Mixed Precision

In [ ]:
# ============================================================
# 11. Training Configuration
# ============================================================

criterion = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)

optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=LEARNING_RATE * 10,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1,
    anneal_strategy='cos'
)

scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

print("Training configuration:")
print(f"  Loss: CTCLoss (blank=0)")
print(f"  Optimizer: AdamW (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})")
print(f"  Scheduler: OneCycleLR (max_lr={LEARNING_RATE * 10})")
print(f"  Mixed Precision: {'Enabled' if torch.cuda.is_available() else 'Disabled'}")

## 11. CTC Decoding & Metrics

In [ ]:
# ============================================================
# 12. CTC Greedy Decoding & Metrics
# ============================================================

def ctc_greedy_decode(log_probs, idx_to_char):
    _, max_indices = torch.max(log_probs, dim=2)
    max_indices = max_indices.permute(1, 0).cpu().numpy()
    
    decoded = []
    for seq in max_indices:
        chars = []
        prev = -1
        for idx in seq:
            if idx != prev:
                if idx != 0:
                    chars.append(idx_to_char.get(idx, '?'))
                prev = idx
        decoded.append(''.join(chars))
    return decoded


def compute_cer(pred, target):
    if len(target) == 0:
        return 1.0 if len(pred) > 0 else 0.0
    m, n = len(pred), len(target)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if pred[i-1] == target[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    return dp[m][n] / n


def compute_wer(pred, target):
    return 0.0 if pred == target else 1.0

print("Decoding and metric functions defined!")

## 12. Training Loop with Mixed Precision & GPU Memory Management

In [ ]:
# ============================================================
# 13. Training & Validation Functions
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch_idx, (images, labels, label_lengths) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        label_lengths = label_lengths.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            log_probs = model(images)
            T, B = log_probs.shape[0], log_probs.shape[1]
            input_lengths = torch.full((B,), T, dtype=torch.long, device=device)
            loss = criterion(log_probs, labels, input_lengths, label_lengths)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return total_loss / max(num_batches, 1)


@torch.no_grad()
def validate(model, loader, criterion, idx_to_char, device):
    model.eval()
    total_loss = 0
    num_batches = 0
    all_cer, all_wer = [], []
    sample_preds = []
    
    for images, labels, label_lengths in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        label_lengths = label_lengths.to(device, non_blocking=True)
        
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            log_probs = model(images)
            T, B = log_probs.shape[0], log_probs.shape[1]
            input_lengths = torch.full((B,), T, dtype=torch.long, device=device)
            loss = criterion(log_probs, labels, input_lengths, label_lengths)
        
        total_loss += loss.item()
        num_batches += 1
        
        preds = ctc_greedy_decode(log_probs, idx_to_char)
        
        for i in range(B):
            gt_indices = labels[i, :label_lengths[i]].cpu().numpy()
            gt_text = ''.join(idx_to_char.get(int(idx), '?') for idx in gt_indices)
            all_cer.append(compute_cer(preds[i], gt_text))
            all_wer.append(compute_wer(preds[i], gt_text))
            if len(sample_preds) < 10:
                sample_preds.append((gt_text, preds[i]))
    
    return total_loss / max(num_batches, 1), np.mean(all_cer), np.mean(all_wer), sample_preds

print("Training and validation functions defined!")

In [ ]:
# ============================================================
# 14. Run Training
# ============================================================

history = {'train_loss': [], 'val_loss': [], 'val_cer': [], 'val_wer': [], 'lr': []}
best_cer = float('inf')
best_epoch = 0
patience = 10
patience_counter = 0

print("=" * 70)
print("Starting Training")
print("=" * 70)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, 
                                  scheduler, scaler, device)
    
    val_loss, val_cer, val_wer, sample_preds = validate(
        model, val_loader, criterion, idx_to_char, device
    )
    
    current_lr = optimizer.param_groups[0]['lr']
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_cer'].append(val_cer)
    history['val_wer'].append(val_wer)
    history['lr'].append(current_lr)
    
    print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"CER: {val_cer:.4f} | "
          f"WER: {val_wer:.4f} | "
          f"LR: {current_lr:.2e}")
    
    if val_cer < best_cer:
        best_cer = val_cer
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_cer': val_cer,
            'val_wer': val_wer,
            'char_to_idx': char_to_idx,
            'idx_to_char': idx_to_char,
            'num_classes': NUM_CLASSES,
        }, os.path.join(OUTPUT_DIR, 'best_model.pth'))
        print(f"  >>> New best CER: {val_cer:.4f} - model saved!")
    else:
        patience_counter += 1
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"\n  Sample predictions (epoch {epoch}):")
        for gt, pred in sample_preds[:5]:
            match = "OK" if gt == pred else "XX"
            print(f"    [{match}] GT: '{gt}' | Pred: '{pred}'")
        print()
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)")
        break
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\n{'=' * 70}")
print(f"Training Complete! Best CER: {best_cer:.4f} at epoch {best_epoch}")
print(f"{'=' * 70}")

## 13. Evaluate Model Performance & Visualize Results

In [ ]:
# ============================================================
# 15. Plot Training Curves
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss', color='blue')
axes[0, 0].plot(history['val_loss'], label='Val Loss', color='red')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('CTC Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['val_cer'], label='Val CER', color='green')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('CER')
axes[0, 1].set_title('Character Error Rate')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['val_wer'], label='Val WER', color='orange')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('WER')
axes[1, 0].set_title('Word Error Rate')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['lr'], label='LR', color='purple')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('LR')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].set_yscale('log')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('BanglaWriting CRNN Training Results', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"Best CER: {best_cer:.4f} (epoch {best_epoch})")
print(f"Final Val WER: {history['val_wer'][-1]:.4f}")
print(f"Word Accuracy: {(1 - history['val_wer'][-1]) * 100:.1f}%")

In [ ]:
# ============================================================
# 16. Visual predictions on validation samples
# ============================================================

checkpoint = torch.load(os.path.join(OUTPUT_DIR, 'best_model.pth'), map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model from epoch {checkpoint['epoch']} (CER: {checkpoint['val_cer']:.4f})")

fig, axes = plt.subplots(4, 4, figsize=(18, 10))
sample_indices = random.sample(range(len(val_samples)), min(16, len(val_samples)))

for ax, idx in zip(axes.flatten(), sample_indices):
    s = val_samples[idx]
    full_img = Image.open(s['img_path']).convert('L')
    x1, y1, x2, y2 = s['bbox']
    word_img = full_img.crop((x1, y1, x2, y2))
    
    w, h = word_img.size
    new_h = IMG_HEIGHT
    new_w = max(1, int(w * new_h / h))
    if new_w > IMG_WIDTH: new_w = IMG_WIDTH
    word_resized = word_img.resize((new_w, new_h), Image.BILINEAR)
    padded = Image.new('L', (IMG_WIDTH, IMG_HEIGHT), 255)
    padded.paste(word_resized, (0, 0))
    
    img_tensor = transforms.Normalize([0.5], [0.5])(transforms.ToTensor()(padded))
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        log_probs = model(img_tensor)
    pred = ctc_greedy_decode(log_probs, idx_to_char)[0]
    
    gt = s['label']
    color = 'green' if pred == gt else 'red'
    ax.imshow(word_img, cmap='gray')
    ax.set_title(f"GT: {gt}\nPred: {pred}", fontsize=9, color=color)
    ax.axis('off')

plt.suptitle('Validation Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'val_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 14. Save Model & Export Artifacts

In [ ]:
# ============================================================
# 17. Save final model, vocab, and training history
# ============================================================

torch.save({
    'epoch': NUM_EPOCHS,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'char_to_idx': char_to_idx,
    'idx_to_char': idx_to_char,
    'num_classes': NUM_CLASSES,
    'history': history,
    'config': {
        'img_height': IMG_HEIGHT,
        'img_width': IMG_WIDTH,
        'hidden_size': 256,
        'num_classes': NUM_CLASSES,
    }
}, os.path.join(OUTPUT_DIR, 'final_model.pth'))

vocab_data = {
    'char_to_idx': char_to_idx,
    'idx_to_char': {str(k): v for k, v in idx_to_char.items()},
    'num_classes': NUM_CLASSES,
}
with open(os.path.join(OUTPUT_DIR, 'vocab.json'), 'w', encoding='utf-8') as f:
    json.dump(vocab_data, f, ensure_ascii=False, indent=2)

history_df = pd.DataFrame(history)
history_df.index.name = 'epoch'
history_df.to_csv(os.path.join(OUTPUT_DIR, 'training_history.csv'))

# Validation predictions export
model.eval()
all_preds, all_gts = [], []
for images, labels, label_lengths in val_loader:
    images = images.to(device)
    with torch.no_grad():
        log_probs = model(images)
    preds = ctc_greedy_decode(log_probs, idx_to_char)
    for i in range(len(preds)):
        gt_indices = labels[i, :label_lengths[i]].numpy()
        gt_text = ''.join(idx_to_char.get(int(idx), '?') for idx in gt_indices)
        all_preds.append(preds[i])
        all_gts.append(gt_text)

results_df = pd.DataFrame({
    'ground_truth': all_gts,
    'prediction': all_preds,
    'correct': [g == p for g, p in zip(all_gts, all_preds)],
    'cer': [compute_cer(p, g) for p, g in zip(all_preds, all_gts)],
})
results_df.to_csv(os.path.join(OUTPUT_DIR, 'val_predictions.csv'), index=False)

print("Saved artifacts:")
print(f"  best_model.pth  - Best model checkpoint")
print(f"  final_model.pth - Final model checkpoint")
print(f"  vocab.json      - Character vocabulary")
print(f"  training_history.csv - Training metrics")
print(f"  val_predictions.csv  - Validation predictions")
print(f"\n{'=' * 70}")
print(f"SUMMARY")
print(f"  Model: CRNN (CNN + BiLSTM + CTC)")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Best CER: {best_cer:.4f}")
print(f"  Word Accuracy: {results_df['correct'].mean() * 100:.1f}%")
print(f"  Mean CER: {results_df['cer'].mean():.4f}")
print(f"{'=' * 70}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 15. Inference Helper - Predict on New Images

Use this function to run inference on any new Bangla handwriting image.

In [ ]:
# ============================================================
# 18. Inference function for production use
# ============================================================

def predict_word(model, image_path_or_pil, device, idx_to_char, 
                 img_height=64, img_width=256):
    model.eval()
    
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil).convert('L')
    else:
        img = image_path_or_pil.convert('L')
    
    w, h = img.size
    new_h = img_height
    new_w = max(1, int(w * new_h / h))
    if new_w > img_width:
        new_w = img_width
    img = img.resize((new_w, new_h), Image.BILINEAR)
    
    padded = Image.new('L', (img_width, img_height), 255)
    padded.paste(img, (0, 0))
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])
    img_tensor = transform(padded).unsqueeze(0).to(device)
    
    with torch.no_grad():
        log_probs = model(img_tensor)
    
    return ctc_greedy_decode(log_probs, idx_to_char)[0]


# --- Usage Example ---
# checkpoint = torch.load('best_model.pth', map_location='cpu')
# model = CRNN(num_classes=checkpoint['num_classes'])
# model.load_state_dict(checkpoint['model_state_dict'])
# model.eval()
# result = predict_word(model, 'word_image.jpg', 'cpu', checkpoint['idx_to_char'])
# print(f"Predicted: {result}")

print("Inference function ready!")
print("\nTo use the trained model later:")
print("  1. Load checkpoint: torch.load('best_model.pth')")
print("  2. Create model: CRNN(num_classes=checkpoint['num_classes'])")
print("  3. Load weights: model.load_state_dict(checkpoint['model_state_dict'])")
print("  4. Predict: predict_word(model, 'image.jpg', device, checkpoint['idx_to_char'])")

## 16. Test Model on Your Own Image

Provide the path to any Bangla handwriting image below to get the model's prediction.  
- For a **full-page** image, the model will crop each word using its JSON annotation.  
- For a **single word crop**, the model predicts directly.

In [ ]:
# ============================================================
# 19. Load Best Model & Test on Your Image
# ============================================================

import os, json, glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# --- Config ---
MODEL_PATH = "best_model.pth"  # Change to "final_model.pth" to use final model
WORKSPACE = "/home/rohan/Softograph/bangla_dataset_1_download"

# Try multiple locations for the model file
for candidate in [
    os.path.join(WORKSPACE, MODEL_PATH),
    os.path.join("/kaggle/working", MODEL_PATH),
    MODEL_PATH,
]:
    if os.path.exists(candidate):
        MODEL_PATH = candidate
        break

print(f"Loading model from: {MODEL_PATH}")
print(f"Model size: {os.path.getsize(MODEL_PATH) / 1024**2:.1f} MB")

# --- Load checkpoint ---
checkpoint = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)
saved_char_to_idx = checkpoint['char_to_idx']
saved_idx_to_char = checkpoint['idx_to_char']
saved_num_classes = checkpoint['num_classes']

print(f"Model from epoch: {checkpoint.get('epoch', '?')}")
print(f"Val CER: {checkpoint.get('val_cer', '?')}")
print(f"Val WER: {checkpoint.get('val_wer', '?')}")
print(f"Vocabulary: {saved_num_classes} classes ({saved_num_classes - 1} chars + blank)")

# --- Rebuild model ---
class BidirectionalLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, output_size)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out)

class CRNN(nn.Module):
    def __init__(self, num_classes, img_height=64, hidden_size=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, 3, 1, 1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, 1, 1), nn.BatchNorm2d(128), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True),
            nn.Conv2d(256, 256, 3, 1, 1), nn.BatchNorm2d(256), nn.ReLU(True), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(256, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True),
            nn.Conv2d(512, 512, 3, 1, 1), nn.BatchNorm2d(512), nn.ReLU(True), nn.MaxPool2d((2,1),(2,1)),
            nn.Conv2d(512, 512, 2, 1, 0), nn.BatchNorm2d(512), nn.ReLU(True),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.Sequential(
            BidirectionalLSTM(512, hidden_size, hidden_size),
            nn.Dropout(0.3),
            BidirectionalLSTM(hidden_size, hidden_size, num_classes),
        )
    def forward(self, x):
        conv = self.cnn(x)
        conv = self.adaptive_pool(conv).squeeze(2).permute(0, 2, 1)
        output = self.rnn(conv).permute(1, 0, 2)
        return F.log_softmax(output, dim=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
infer_model = CRNN(num_classes=saved_num_classes, img_height=64, hidden_size=256).to(device)
infer_model.load_state_dict(checkpoint['model_state_dict'])
infer_model.eval()
print(f"\nModel loaded on {device} — ready for inference!")
print("="*50)

In [ ]:
# ============================================================
# 20. Test on YOUR image — change TEST_IMAGE_PATH below
# ============================================================

# >>>>>>>>>> CHANGE THIS PATH TO YOUR IMAGE <<<<<<<<<<
# On Kaggle: upload your test image as a dataset or use a dataset image
TEST_IMAGE_PATH = "/kaggle/input/banglawriting/converted/converted/0_21_0.jpg"
# Local: "/home/rohan/Softograph/bangla_dataset_1_download/test_image.png"

IMG_HEIGHT, IMG_WIDTH = 64, 256

def ctc_decode(log_probs, idx_to_char):
    _, max_idx = torch.max(log_probs, dim=2)
    max_idx = max_idx.permute(1, 0).cpu().numpy()
    decoded = []
    for seq in max_idx:
        chars, prev = [], -1
        for idx in seq:
            if idx != prev:
                if idx != 0:
                    chars.append(idx_to_char.get(idx, '?'))
                prev = idx
        decoded.append(''.join(chars))
    return decoded

def preprocess_word_crop(img, img_height=64, img_width=256):
    """Resize word image, maintaining aspect ratio, pad to fixed size."""
    if img.mode != 'L':
        img = img.convert('L')
    w, h = img.size
    new_h = img_height
    new_w = max(1, int(w * new_h / h))
    if new_w > img_width:
        new_w = img_width
    img = img.resize((new_w, new_h), Image.BILINEAR)
    padded = Image.new('L', (img_width, img_height), 255)
    padded.paste(img, (0, 0))
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]),
    ])
    return transform(padded).unsqueeze(0)

def predict_single_image(model, img, device, idx_to_char):
    """Predict text from a single word-crop PIL image."""
    tensor = preprocess_word_crop(img).to(device)
    with torch.no_grad():
        log_probs = model(tensor)
    return ctc_decode(log_probs, idx_to_char)[0]

# --- Check if the image exists ---
assert os.path.exists(TEST_IMAGE_PATH), f"Image not found: {TEST_IMAGE_PATH}"
print(f"Testing image: {TEST_IMAGE_PATH}")

# --- Check if there's a matching JSON annotation ---
base = os.path.splitext(TEST_IMAGE_PATH)[0]
json_path = base + ".json"

if os.path.exists(json_path):
    # Full-page image with annotations — crop each word and predict
    print(f"Found annotation: {json_path}")
    with open(json_path, 'r', encoding='utf-8') as f:
        annotation = json.load(f)
    
    full_img = Image.open(TEST_IMAGE_PATH).convert('L')
    words = annotation['shapes']
    print(f"Words annotated: {len(words)}\n")
    
    # Show full page
    fig_full, ax_full = plt.subplots(1, 1, figsize=(12, 16))
    ax_full.imshow(Image.open(TEST_IMAGE_PATH), cmap='gray')
    ax_full.set_title(f"Full Page: {os.path.basename(TEST_IMAGE_PATH)}", fontsize=14)
    ax_full.axis('off')
    plt.tight_layout()
    plt.show()
    
    # Predict each word crop
    n_words = len(words)
    cols = 4
    rows = min((n_words + cols - 1) // cols, 10)  # Max 10 rows
    fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 2.5))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    correct, total = 0, 0
    for i, shape in enumerate(words):
        if i >= rows * cols:
            break
        gt_label = shape['label'].strip()
        pts = shape['points']
        x1, y1 = int(pts[0][0]), int(pts[0][1])
        x2, y2 = int(pts[1][0]), int(pts[1][1])
        x1, x2 = max(0, min(x1, x2)), min(full_img.width, max(x1, x2))
        y1, y2 = max(0, min(y1, y2)), min(full_img.height, max(y1, y2))
        
        if x2 - x1 < 5 or y2 - y1 < 5:
            continue
        
        word_crop = full_img.crop((x1, y1, x2, y2))
        pred = predict_single_image(infer_model, word_crop, device, saved_idx_to_char)
        
        is_correct = pred == gt_label
        if is_correct:
            correct += 1
        total += 1
        
        r, c = i // cols, i % cols
        ax = axes[r, c]
        ax.imshow(word_crop, cmap='gray')
        color = 'green' if is_correct else 'red'
        ax.set_title(f"GT: {gt_label}\nPred: {pred}", fontsize=9, color=color)
        ax.axis('off')
    
    # Hide unused axes
    for i in range(total, rows * cols):
        r, c = i // cols, i % cols
        axes[r, c].axis('off')
    
    plt.suptitle(f"Predictions on {os.path.basename(TEST_IMAGE_PATH)} — "
                 f"Accuracy: {correct}/{total} ({correct/max(total,1)*100:.1f}%)", 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\nResults: {correct}/{total} words correct ({correct/max(total,1)*100:.1f}% accuracy)")

else:
    # Single word crop image — predict directly
    print("No JSON annotation found — treating as a single word crop image")
    img = Image.open(TEST_IMAGE_PATH)
    pred = predict_single_image(infer_model, img, device, saved_idx_to_char)
    
    fig, ax = plt.subplots(1, 1, figsize=(8, 3))
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Prediction: {pred}", fontsize=16, color='blue', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    print(f"\n>>> Predicted text: {pred}")